# FAQ Retrieval Eval

Check ingestion and retrieval via Weaviate.

In [1]:
import sys
sys.path.append("../")

from dotenv import load_dotenv
from llama_index.core import Settings, StorageContext, VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.weaviate import WeaviateVectorStore

from ingestion.faq.faq_doc_builder import build_llama_documents
from ingestion.faq.faq_chunker import split_by_docs
from src.core.transformers import ConditionalFixedChunker
from src.core.utils import ingest_pipeline
from vectorstore.weaviate import client

In [2]:
load_dotenv(override=True)
embedding_model = OpenAIEmbedding(model="text-embedding-3-large")
Settings.embed_model = embedding_model

/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
sections = split_by_docs("../data/raw/faq.docx")
documents = build_llama_documents(sections)
nodes = ingest_pipeline(
    documents,
    transformations=[ConditionalFixedChunker(), embedding_model],
)
len(nodes)

101

In [4]:
weaviate_client = client.local()

In [5]:
if weaviate_client.collections.exists("Faq"):
    weaviate_client.collections.delete("Faq")

vector_store = WeaviateVectorStore(weaviate_client=weaviate_client, index_name="Faq")
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex(nodes=nodes, storage_context=storage_context)

In [6]:
retriever = index.as_retriever(similarity_top_k=3)
results = retriever.retrieve("How to register with an email?")

for node in results:
    print(node.text)
    print(node.metadata)
    print("score:", getattr(node, "score", None))
    print("---")

How to register with an email
Open the registration page. Choose whether you want to hold a regular account or a business account.
Enter your email address and create a password.
Determine whether you are over 18 years old. If not, provide your exact date of birth. You will hold a Junior account which will be transformed into a regular account once you turn 18.
Accept the HopShop Terms & Conditions and click [register].
You will receive a confirmation email. Open your mailbox and click (confirm registration) to complete the registration process.
Once you complete the registration, we will redirect you to your new HopShop account. We will give you a default username ― you can change it in the Account details tab.
{'section_title': 'How to register with an email', 'source_type': 'faq', 'source_file': 'faq.docx', 'section_id': 2}
score: 1.0
---
How to register with Google or Facebook account
Open the sign in page.
Select signing in through Facebook or Google.
Sign in to your Facebook or G

In [ ]:
weaviate_client.close()